# Results figures

Every figure of thesis section 7, regenerated from the cached data produced by NB1-9 and the cluster sweeps. No transfer matrices are built here; each cell loads the caches, redoes the fits, and saves one figure. Rerunning the notebook after new cluster rungs land re-plots everything.

In [ ]:
include("../src/thesislib.jl")
using JLD2, Printf, Statistics, LsqFit
thesis_plot_theme!()

const V = load("../results/data/alcaraz_velocity.jld2", "v")
const M8 = load("../results/data/nb8_master.jld2", "done")
const CL = "../results/data/cluster"
const PVALS = (0.0, 0.1, 0.3, 0.5)   # the couplings the thesis reports

# chord variable and helpers used by every entropy fit
W(t, T) = log((2T / pi) * sin(pi * t / T))
ph(z) = angle(-z)
dphw(a, b) = mod(a - b + pi, 2pi) - pi

load_arm(file, label) = Dict(k[2] => v for (k, v) in load(joinpath(CL, file), "done") if k[1] == label && !haskey(v, :error))

# the corrected column carries a _bulk suffix in the label; at p=0 both columns are the same operator
suffix(p) = p == 0.0 ? "" : "_bulk"
# p=1.0 is the halved-Trotter arm, so its label carries the step
ent_arm(p) = p == 1.0 ? load_arm("sweep_ent_p1.0_bulk_dt0.05.jld2", "ent_p1.0_bulk_dt0.05") : load_arm("sweep_ent_p$(p)$(suffix(p)).jld2", "ent_p$(p)$(suffix(p))")
eig_arm(p) = load_arm("sweep_rtm_eigs_p$(p)$(suffix(p)).jld2", "rtm_eigs_p$(p)$(suffix(p))")
# the half-integer ladder merged into the integer one wherever the cluster has run it, since the
# Eq.(3) fit is limited by the number of points inside the window rather than by its length
function spec_arm(p)
    if p >= 0.3
        sfx = p == 1.0 ? "_bulk_dt0.05" : "_bulk"
        return merge(ent_arm(p), load_arm("sweep_tower_p$(p)$(sfx).jld2", "tower_p$(p)$(sfx)"))
    end
    a = eig_arm(p)
    f = "sweep_rtm_eigs_p$(p)_fine$(suffix(p)).jld2"
    isfile(joinpath(CL, f)) || return a
    return merge(a, load_arm(f, "rtm_eigs_p$(p)_fine$(suffix(p))"))
end
tower_arm(p, sfx="_bulk") = load_arm("sweep_tower_p$(p)$(sfx).jld2", "tower_p$(p)$(sfx)")

lin(x, q) = q[1] .* x .+ q[2]

# slope of one profile against the chord variable, over the middle half of the cuts
function chord_slope(s2, T)
    s2r = real.(s2)
    n = length(s2r)
    ts = range(T / (n + 1), T - T / (n + 1), length=n)
    bulk = (n ÷ 4):(3n ÷ 4)
    xs = W.(ts[bulk], T)
    ys = s2r[bulk]
    f = curve_fit(lin, xs, ys, [0.06, 0.5])
    return f.param[1], xs, ys
end

# the interior of the imaginary profile, which the prediction says is flat
Tlab(T) = T == round(T) ? string(Int(T)) : string(T)

plateau(s2) = mean(imag.(s2)[max(1, length(s2) ÷ 2 - 1):(length(s2) ÷ 2 + 2)])

# times over which the profile still follows the chord line; past them the power method has left
# the physical branch and the profile is an artefact
ent_max = Dict(0.0 => 15.0, 0.1 => 9.0, 0.3 => 6.0, 0.5 => 5.0)

# up to four times from the upper half of the clean window; short arms give their last four
function upper4(p)
    Ts = sort([T for T in keys(ent_arm(p)) if T <= ent_max[p]])
    top = [T for T in Ts if T >= (first(Ts) + last(Ts)) / 2]
    length(top) < 4 && (top = Ts[max(1, end-3):end])
    return top[round.(Int, range(1, length(top), length=min(4, length(top))))]
end

# one row of the entropy figures: Re S2 with the chord curves, Im S2 with the plateau line
function dome_column(armd, Tshow, plab, panels)
    pal = cgrad(:viridis, length(Tshow), categorical=true)
    pa = plot(xlabel="t/T", ylabel="Re S₂", legend=false,
              title="($(panels[1])) p = $plab")
    pb = plot(xlabel="t/T", ylabel="Im S₂", legend=:outerright, xticks=[0.0, 0.5, 1.0],
              title="($(panels[2])) p = $plab")
    for (i, T) in enumerate(Tshow)
        s2 = armd[T].s2_base
        n = length(s2)
        ts = range(T / (n + 1), T - T / (n + 1), length=n)
        bulk = (n ÷ 4):(3n ÷ 4)
        # s0 is non-universal, so match it to the data and compare shapes only
        s0 = mean(real.(s2)[bulk] .- (0.5 / 8) .* W.(ts[bulk], T))
        scatter!(pa, ts ./ T, real.(s2), color=pal[i], ms=3, msw=0, label="T=$(Tlab(T))")
        tt = range(0.04, 0.96, length=200)
        plot!(pa, tt, (0.5 / 8) .* W.(tt .* T, T) .+ s0, color=pal[i], lw=1.2, ls=:dash, label="")
        scatter!(pb, ts ./ T, imag.(s2), color=pal[i], ms=3, msw=0, label="T=$(Tlab(T))")
    end
    hline!(pb, [pi / 32], color=:black, lw=2.0, ls=:dash, label="πc/16")
    return pa, pb
end

mkpath("../results/imgs")
println("caches loaded; v(p) = ", V)
for p in PVALS
    e = ent_arm(p)
    Ts = sort(collect(keys(e)))
    @printf("ent p=%.1f: T=%g-%g (%d rungs), clean to T=%g\n", p, first(Ts), last(Ts), length(Ts), ent_max[p])
end

## The entropy route at the integrable point

Prediction: the generalized Renyi-2 temporal entropy follows the chord profile with slope c/8 in the real part, and its imaginary part is flat at pi c/16. The cut is a boundary cut, so the coefficient is c/8 and not the bulk c/3.

The cell plots both parts at four evolution times taken from the upper half of each clean window, for p=0 and p=0.1. The constant s0 is non-universal, so it is matched to the data and only the shape is compared, which is the comparison made in the reference.

In [ ]:
# fig:domes -- the non-integrable coupling the main text shows, Re beside Im
T1 = upper4(0.1)
pa, pb = dome_column(ent_arm(0.1), T1, "0.1", ("a", "b"))

fig = plot(pa, pb, layout=@layout([a b{0.545w}]), size=thesis_size(0.95; aspect=0.36),
           margin=3Plots.mm, bottom_margin=6Plots.mm, left_margin=7Plots.mm)
savefig(fig, "../results/imgs/res_domes.png")

for T in T1
    @printf("p=0.1 T=%-4.1f plateau = %.4f   excess over πc/16 = %+.4f\n",
            T, plateau(ent_arm(0.1)[T].s2_base), plateau(ent_arm(0.1)[T].s2_base) - pi/32)
end
fig

The real part collapses onto the chord curves at c=1/2 in both columns. The imaginary part is flat across the interior of the strip and sits above pi c/16 = 0.0982 throughout: 0.1241, 0.1249, 0.1170, 0.1175 at p=0 for T=9, 11, 13, 15, and 0.1285, 0.1307, 0.1287, 0.1258 at p=0.1 for T=6 to 9.

The plateau approaches the asymptote from above and falls slowly, which is the finite-time correction. Reading c from these times alone would give 0.60 to 0.67; the asymptotic value requires the extrapolation of the next notebook cell and of section 7.1.

## The entropy route at larger coupling

The window over which the profile stays conformal shrinks as p grows. The cell plots the real part against the chord variable, where a conformal profile is a straight line, and the plateau against T, which should fall smoothly towards pi c/16.

This is the appendix figure. It shows where the route gives out rather than a measurement.

In [ ]:
# app:highp -- the couplings the main text does not show, in the layout of fig:domes
hi = [(0.0, "0", ("a", "b")), (0.3, "0.3", ("c", "d")), (0.5, "0.5", ("e", "f"))]

rows = Plots.Plot[]
for (p, plab, tags) in hi
    a = ent_arm(p)
    Ts = sort([T for T in keys(a) if T <= ent_max[p]])
    # at most six times per row, evenly spaced, so the legend stays clear of the axis
    length(Ts) > 6 && (Ts = Ts[round.(Int, range(1, length(Ts), length=6))])
    pa, pb = dome_column(a, Ts, plab, tags)
    push!(rows, pa)
    push!(rows, pb)
end

fig = plot(rows..., layout=@layout([a b{0.545w}; c d{0.545w}; e f{0.545w}]),
           size=thesis_size(1.0; aspect=0.95), margin=3Plots.mm,
           bottom_margin=6Plots.mm, left_margin=7Plots.mm)
savefig(fig, "../results/imgs/res_domes_hi.png")
fig

At p=0.3 the plateau falls from 0.1437 at T=2 to 0.1323 at T=4 and then stops falling: 0.1328 at T=5 and 0.1320 at T=6. At p=0.5 it is not even monotone, 0.1460, 0.1341, 0.1372, 0.1244.

Four or five rungs, over which the plateau has largely stopped moving, do not determine the asymptote of a T^(-1/2) decay. This is why no entropy-based central charge is quoted beyond p=0.1.

## Where the eigenvector route ends

The route breaks somewhere, and the phase rigidity of the dominant pair is one candidate for what sets it. Panel a shows the rigidity against T for every p with eigenvector data. Panel b shows the p=0.1 chord scatter with the past-wall points included: the T > 9 points leave the line the clean window defines.

In [ ]:
# panel a: rigidity of the physical pair against T, per coupling
pa = plot(xlabel="T", ylabel="phase rigidity r", yscale=:log10, legend=:bottomleft)
for p in PVALS
    rig = Dict(T => e.rigidity[e.i0] for (T, e) in ent_arm(p) if !isempty(e.rigidity))
    Ts = sort(collect(keys(rig)))
    isempty(Ts) && continue
    plot!(pa, Ts, [rig[T] for T in Ts], color=P_COLOR[p], marker=P_MARKER[p], ms=5, label="p=$(p)")
    @printf("p=%.1f  r: %.2g at T=%g  ->  %.2g at T=%g\n", p, rig[Ts[1]], Ts[1], rig[Ts[end]], Ts[end])
end

# panel b: the chord scatter with the points past the window included, which is where they leave
# the line the window defines
pb = plot(xlabel="W(t,T)", ylabel="Re S₂", legend=:topleft, ylims=(0.1, 0.78))
fits = Dict{Float64,Vector{Float64}}()
for p in (0.0, 0.1)
    armd = ent_arm(p)
    xs = Float64[]
    ys = Float64[]
    firstlab = true
    for T in sort(collect(keys(armd)))
        _, wx, wy = chord_slope(armd[T].s2_base, T)
        scatter!(pb, wx, wy, color=P_COLOR[p], marker=P_MARKER[p], ms=3.5, msw=0, alpha=0.7,
                 label=firstlab ? "p=$(p)" : "")
        firstlab = false
        if 4.0 <= T <= ent_max[p]
            append!(xs, wx)
            append!(ys, wy)
        end
    end
    f = curve_fit(lin, xs, ys, [0.06, 0.5])
    fits[p] = f.param
    @printf("p=%.1f  pooled over T=4-%g: slope=%.4f  c=8s=%.3f  (%d points)\n",
            p, ent_max[p], f.param[1], 8 * f.param[1], length(xs))
end
for p in (0.0, 0.1)
    wr = range(-0.35, 2.2, length=40)
    plot!(pb, wr, lin(wr, fits[p]), color=:black, ls=:dash, label=p == 0.1 ? "conformal fits" : "")
end

fig = plot(pa, pb, layout=(1, 2), size=(1150, 430), margin=5Plots.mm,
           bottom_margin=10Plots.mm, left_margin=10Plots.mm)
savefig(fig, "../results/imgs/res_wall.png")
fig

The rigidity falls geometrically at every coupling and faster as p grows, from 0.29 to 5e-8 over T=2 to 18 at p=0 and from 0.046 to 5.6e-7 over T=2 to 9 at p=0.1. It spans four orders of magnitude between couplings at the point where each route stops, so it does not identify a threshold.

Notebook 8 carries the corrected reading of the fall itself: per temporal site it is flat, so these curves are the extensive decay of an overlap rather than a growing pathology of the transfer matrix. The pooled chord fits give c=8s of 0.645 at p=0 and 0.680 at p=0.1, both offset from 1/2 by the same finite-time bias the plateau shows.

## The profile one step past the window

A conformal profile is linear in W with slope c/8. The cell takes the last time inside the window and the first one past it, at the coupling whose cache still holds a rung past its window, and plots both against that line with the cuts from the two halves of the strip marked separately.

In [ ]:
# the last rung inside the window against the first one past it, at the coupling whose cache
# still holds a rung past its window
pw = first(p for p in PVALS if any(T > ent_max[p] for T in keys(ent_arm(p))))
armw = ent_arm(pw)
Tpair = (ent_max[pw], minimum(T for T in keys(armw) if T > ent_max[pw]))

panels = []
for T in Tpair
    s2 = real.(armw[T].s2_base)
    n = length(s2)
    ts = range(T / (n + 1), T - T / (n + 1), length=n)
    Ws = W.(ts, T)
    first_half = [i for i in 1:n if ts[i] <= T / 2]
    second_half = [i for i in 1:n if ts[i] > T / 2]

    # conformal line: slope c/8 at c=1/2, offset fixed on the small-W points
    small_W = [i for i in 1:n if Ws[i] < 0.5]
    s0 = mean(s2[small_W] .- (0.5 / 8) .* Ws[small_W])
    Wr = range(minimum(Ws), maximum(Ws), length=50)

    pan = plot(xlabel="W(t,T)", ylabel=(T == Tpair[1] ? "Re S₂" : ""),
               legend=(T == Tpair[1] ? :topleft : false), title="T = $(Int(T))", titlefontsize=11)
    plot!(pan, Wr, (0.5 / 8) .* Wr .+ s0, color=:black, ls=:dash, lw=1.5, label="c = 1/2")
    scatter!(pan, Ws[first_half], s2[first_half], color=:dodgerblue, marker=:circle, ms=5, msw=0, label="t < T/2")
    scatter!(pan, Ws[second_half], s2[second_half], color=:crimson, marker=:diamond, ms=5, msw=0, label="t > T/2")
    push!(panels, pan)

    deviation = maximum(abs.(s2 .- ((0.5 / 8) .* Ws .+ s0)))
    n_maxima = count(i -> 1 < i < n && s2[i] > s2[i-1] && s2[i] > s2[i+1], 1:n)
    @printf("p=%.1f T=%g  peak Re S2 = %.3f  n_maxima = %d  max deviation from the c=1/2 line = %.3f\n",
            pw, T, maximum(s2), n_maxima, deviation)
end
fig = plot(panels..., layout=(1, 2), size=(1100, 430), margin=5Plots.mm,
           bottom_margin=10Plots.mm, left_margin=10Plots.mm)
savefig(fig, "../results/imgs/res_broken_dome.png")
fig

At T=15 the profile stays on the conformal line, with a maximum deviation of 0.046 and a peak of 0.344. At T=16 the peak is 4.03 and the deviation 3.66, two orders of magnitude larger.

The break is a step, not a drift. Past it the iteration is no longer converging to the physical eigenvectors, so the shape of the T=16 curve carries no physical meaning and only the time of the departure is used.

## The spectral route at the integrable point

The eigenvalue-only ladder reaches T=20 at p=0. Emergent dual unitarity predicts that mu0 keeps a constant modulus while its phase winds, so normalising by the modulus should place every time on the unit circle.

In [ ]:
# fig:circle -- mu0 at every coupling, each ladder divided by its own mean modulus. Normalising by
# each point's own modulus would put it on the circle whatever the data did; dividing by one
# reference per coupling keeps a drift visible as a radial departure.
spec = [(p, spec_arm(p)) for p in PVALS]

fig = plot(xlabel="Re μ₀/μ̄", ylabel="Im μ₀/μ̄", aspect_ratio=:equal,
           legend=(0.42, 0.615), size=thesis_size(0.44; aspect=0.95),
           xlims=(-1.28, 1.28), ylims=(-1.28, 1.28),
           xticks=[-1, 0, 1], yticks=[-1, 0, 1],
           margin=0Plots.mm, top_margin=1Plots.mm, right_margin=2Plots.mm,
           bottom_margin=2Plots.mm, left_margin=2Plots.mm)
th = range(0, 2pi, length=400)
plot!(fig, cos.(th), sin.(th), color=:gray, lw=1.0, ls=:dash, label="")
for (n, (p, a)) in enumerate(spec)
    Ts = sort(collect(keys(a)))
    mus = [a[T].theta_phys for T in Ts]
    mbar = mean(abs.(mus))
    z = mus ./ mbar
    scatter!(fig, real.(z), imag.(z), ms=4, msw=0, color=n, label="p = $p")
    @printf("p=%.1f  T=%g-%g  |μ₀| = %.4f to %.4f  spread %.2f%%\n", p, first(Ts), last(Ts),
            minimum(abs.(mus)), maximum(abs.(mus)), 100*(maximum(abs.(mus))-minimum(abs.(mus)))/mbar)
end
savefig(fig, "../results/imgs/res_circle.png")
fig

The phase winds through the whole ladder at both couplings, and the modulus holds to within one per cent: 1.4825 to 1.4963 over T=2 to 20 at p=0, a spread of 0.93 per cent, and 1.5549 to 1.5619 over T=2 to 9 at p=0.1, a spread of 0.45 per cent.

Panel (b) shows that the residual variation is a slow monotone decrease rather than scatter, so the modulus is constant only to the accuracy of the finite-time corrections, not exactly. Panel (a) is normalised by one reference modulus per coupling rather than by each point's own, so a drift of this kind would show as a radial departure from the circle; at this scale 1 per cent sits inside the marker.

In [ ]:
# fig:x1_p0 -- the boundary exponent for the two boundary conditions, one panel each because the
# targets are 1/2 and 2 and a shared axis hides the approach of the smaller one
free_d = spec_arm(0.0)   # both panels sit on the same evolution times, T=3 upward
fixed_raw = load("../results/data/nb7_fixedbc_k8.jld2", "done")
fixed_d = Dict(k[2] => v for (k, v) in fixed_raw if k[1] == 0.0 && !haskey(v, :error))

# signed phase gap to the block member nearest in phase to the physical branch
function gap1(e)
    ph0 = ph(e.theta[e.i0])
    best, bg = 0.0, Inf
    for (i, t) in enumerate(e.theta)
        i == e.i0 && continue
        g = dphw(ph(t), ph0)
        abs(g) < bg && (bg = abs(g); best = g)
    end
    return best
end

@. gap_model(T, q) = q[1] / T + q[2] / T^3

Tlo = 3.0
Thi = min(maximum(keys(free_d)), maximum(keys(fixed_d)))

function x1_panel(dat, col, mk, target, label, title, showx)
    Ts = [T for T in sort(collect(keys(dat))) if Tlo <= T <= Thi]
    gaps = [gap1(dat[T]) for T in Ts]
    xs = V[0.0] .* Ts .* gaps ./ pi

    f = curve_fit(gap_model, Ts, gaps, [1.0, 0.0])
    x1 = V[0.0] * f.param[1] / pi

    pl = plot(xlabel="T", ylabel="x₁", title=title, titlefontsize=10,
              legend=:bottomright, xlims=(Tlo - 0.4, Thi + 0.4))
    hline!(pl, [target], color=:gray, ls=:dash, lw=1.2, label="")
    Tr = range(Tlo, Thi, length=120)
    plot!(pl, Tr, V[0.0] .* Tr .* gap_model(Tr, f.param) ./ pi, color=col, lw=1.4,
          label=@sprintf("fit, x₁ = %.3f", x1))
    scatter!(pl, Ts, xs, color=col, marker=mk, ms=4.5, msw=0, label=label)

    @printf("%-12s T=%g..%g  n=%d  x1=%.4f  (per-time %.3f..%.3f)\n",
            label, Tlo, Thi, length(Ts), x1, minimum(xs), maximum(xs))
    return pl
end

pa = x1_panel(free_d, 1, :circle, 0.5, "free |X+⟩", "(a) free boundary", true)
pb = x1_panel(fixed_d, 2, :square, 2.0, "fixed |Up⟩", "(b) fixed boundary", true)

fig = plot(pa, pb, layout=(1, 2), size=thesis_size(0.9; aspect=0.44), margin=4Plots.mm,
           left_margin=9Plots.mm, bottom_margin=8Plots.mm)
savefig(fig, "../results/imgs/res_x1_p0.png")
fig

The first spectral gap converted to a dimension gives 0.497 for the free boundary and 1.996 for the fixed one, against the exact 1/2 and 2.

Each panel has its own scale, because on a shared axis the free branch is a flat line. The free points run from 0.452 at T=2 to 0.503 at T=9 with a residual scatter of about 0.01, within which the last of them lies just above 1/2; the fixed points rise from 1.819 to 1.978 and stay below their target throughout.

## The central charge from the phase

Eq. (3) gives Im lambda0 / T = a0 + B/T + C/T^2, with c = 24 v |C| / pi. The branch constant B is known to be -pi, and the cell leaves it fixed at that value.

The phase is unwrapped over the whole ladder before the fit. Unwrapping inside a window discards the absolute branch and the fit then returns a central charge in the hundreds.

In [ ]:
# fig:eq3 -- the Eq.(3) fit of the unwrapped phase, one panel per coupling
# unwrap over the whole ladder before any window is applied, or the absolute branch is lost
function unwrapped(mus)
    phi = [ph(m) for m in mus]
    for i in 2:length(phi)
        phi[i] = phi[i - 1] + dphw(phi[i], phi[i - 1])
    end
    return phi
end

@. eq3(x, q) = q[1] - pi / x + q[2] / x^2

panels = Plots.Plot[]
for (n, (p, a)) in enumerate([(q, spec_arm(q)) for q in PVALS])
    Ts = sort(collect(keys(a)))
    phi = unwrapped([a[T].theta_phys for T in Ts])
    y = phi ./ Ts
    f = curve_fit(eq3, Ts, y, [0.1, -0.01])
    cval = 24 * V[p] * abs(f.param[2]) / pi

    pl = plot(xlabel=(n >= 3 ? "T" : ""), ylabel=(isodd(n) ? "Im λ₀/T − a₀" : ""),
              legend=:bottomright, title=@sprintf("p = %.1f", p))
    scatter!(pl, Ts, y .- f.param[1], ms=4, msw=0, color=n, label="")
    tt = range(first(Ts), last(Ts), length=200)
    plot!(pl, tt, eq3(tt, f.param) .- f.param[1], color=n, lw=1.4,
          label=@sprintf("c = %.2f", cval))
    push!(panels, pl)

    @printf("p=%.1f  %2d rungs T=%.0f-%.0f  a₀=%.4f  |C|=%.5f  c=%.4f\n",
            p, length(Ts), first(Ts), last(Ts), f.param[1], abs(f.param[2]), cval)
end

fig = plot(panels..., layout=(2, 2), size=thesis_size(0.95; aspect=0.60),
           margin=3Plots.mm, left_margin=8Plots.mm, bottom_margin=7Plots.mm)
savefig(fig, "../results/imgs/res_eq3.png")
fig

The fit gives c = 0.451 at p=0 over 28 rungs from T=2 to 20 and c = 0.465 at p=0.1 over 10 rungs from T=2 to 9, with |C| = 0.0295 and 0.0228. Both agree to two decimals with the table of section 7, which is fitted on the denser local ladders.

At p=0 the free-fermion value of the Casimir coefficient is pi/96 = 0.0327, so the measured coefficient is 10 per cent low and the central charge with it. Both couplings sit a few per cent below 1/2 rather than above it, which is the opposite sign to the entropy route and is why the two are quoted separately in section 7.

Adding the half-integer rungs moves p=0 from 0.448 to 0.451 without extending the window, which is the density effect the section describes.

## The boundary tower

Prediction: the phase gaps of the k=8 block, converted through x = vT|dphi|/pi, reproduce the
Ising boundary tower 1/2, 3/2, 2. One panel per coupling, with the Ising values dashed.

The members are ordered by phase distance from the physical branch, which is the ordering the
tower itself predicts. Only the first three are quoted; the higher members of the block are not
resolved at these times.

In [ ]:
# fig:tower -- boundary dimensions of each rung against the Ising values
function dims(e, v)
    gaps = sort([abs(dphw(ph(t), ph(e.theta[e.i0]))) for (i, t) in enumerate(e.theta) if i != e.i0])
    return v .* e.T .* gaps ./ pi
end

# p=1.0 needs the halved Trotter step, so its arm carries a different suffix
# p=0 has no k=8 tower arm yet, so its panel comes from the k=4 eigenvalue ladder
sources = [(0.0, :spec), (0.1, :tower), (0.3, :tower), (0.5, :tower)]
marks = [:circle, :square, :diamond]

panels = Plots.Plot[]
for (n, (p, kind)) in enumerate(sources)
    armt = kind === :spec ? eig_arm(p) : tower_arm(p)
    Ts = sort(collect(keys(armt)))
    pl = plot(xlabel=(n >= 3 ? "T" : ""), ylabel=(isodd(n) ? "x" : ""), legend=false,
              title="p = $p", ylims=(0.0, 3.15), xticks=(floor(Int,first(Ts)):max(1, round(Int, (last(Ts)-first(Ts))/4)):ceil(Int,last(Ts))))
    for target in (0.5, 1.5, 2.0)
        hline!(pl, [target], color=:black, lw=1.0, ls=:dash, label="")
    end
    for d in 1:3
        plot!(pl, Ts, [dims(armt[T], V[p])[d] for T in Ts], color=d, marker=marks[d], ms=4, msw=0,
              lw=1.2, label="")
    end
    push!(panels, pl)

    for T in Ts
        @printf("p=%.1f T=%-4.1f %s\n", p, T, join([@sprintf("%.3f", x) for x in dims(armt[T], V[p])[1:3]], "  "))
    end
end

fig = plot(panels..., layout=(2, 2), size=thesis_size(0.95; aspect=0.60), margin=5Plots.mm,
           bottom_margin=9Plots.mm, left_margin=9Plots.mm)
savefig(fig, "../results/imgs/res_tower.png")
fig

The first three members land on the Ising boundary tower at every coupling. At the longest time of each arm they are 0.497, 1.496, 1.810 at p=0.1; 0.493, 1.498, 2.004 at p=0.3; 0.488, 1.511, 2.033 at p=0.5; and 0.494, 1.568, 2.140 at p=1.0. The higher members approach 3/2 and 2 from above as T grows, while x1 sits at 1/2 throughout.

Two rungs misassign a member: p=0.1 at T=2 gives 0.826 and 1.470, and p=0.3 at T=3 gives 1.053 and 1.555. Both are ordered by phase distance, and at those times the members are close enough that the ordering swaps. They are left visible rather than repaired, since the repair would be a choice about which state is which.